In [1]:
# NB04_diag_correlation.ipynb

# %% [markdown]
# # NB04_diag – Correlation Diagnostic for Independent DGM Design
# **Purpose:** Identify financial variables that are weakly correlated with
# pd_system, to serve as the driver of upgrade probability in the
# independent data-generating mechanism (NB04b).
#
# **Why this matters:**
# In the naive DGM (current NB04), upgrade probability is a deterministic
# function of system_grade, which is itself a quantile transform of pd_system.
# Because default risk also increases monotonically with pd_system, the
# upgrade group mechanically over-samples high-risk firms. To break this
# circularity, the independent DGM must drive upgrades using a signal that
# is (i) weakly correlated with pd_system and, ideally, (ii) weakly correlated
# with the default outcome, so that the induced selection is not mechanical.
#
# **Input:**  `data/processed/graded_data.parquet`
# **Output:** console diagnostics only (no files written)

# %%
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROC_DIR = "../data/processed/"
graded = pd.read_parquet(PROC_DIR + "graded_data.parquet")
print(f"Loaded: {graded.shape}")
print(f"Columns: {list(graded.columns)[:20]} ...")

# %% [markdown]
# ## 1. Configure column names
# Adjust these to match the actual dataset if a KeyError is raised.

# %%
PD_COL      = "pd_system"   # system-estimated default probability
DEFAULT_COL = "default"     # binary default label

# Candidate financial variables: raw attributes + named ratios
financial_cols = [
    c for c in graded.columns
    if c.startswith("Attr")
    or c in [
        "log_total_assets", "sales_growth",
        "retained_earnings_to_assets", "ebit_to_assets",
        "net_profit_to_assets", "equity_to_assets",
    ]
]
# Exclude any engineered/system columns that may share Attr-like prefixes
financial_cols = [c for c in financial_cols
                  if c not in [PD_COL, DEFAULT_COL,
                               "system_grade", "grade_ordinal"]]

print(f"\nCandidate financial variables: {len(financial_cols)}")

# %% [markdown]
# ## 2. Correlation with pd_system (Spearman, to capture monotonic links)
# Low |correlation| => good candidate for driving independent upgrades.

# %%
corr_with_pd = (
    graded[financial_cols]
    .apply(lambda x: x.corr(graded[PD_COL], method="spearman"))
    .abs()
    .sort_values()
)

print("=== LOWEST |corr| with pd_system (bottom 15) ===")
print("-> best candidates for the independent DGM upgrade driver")
print(corr_with_pd.head(15).round(4).to_string())

print("\n=== HIGHEST |corr| with pd_system (top 10) ===")
print("-> already embedded in pd_system; exclude as drivers")
print(corr_with_pd.tail(10).round(4).to_string())

# %% [markdown]
# ## 3. Correlation of low-pd candidates with the default outcome
# A candidate that is ALSO weakly correlated with default lets us construct
# an upgrade rule whose induced selection is close to random with respect to
# risk. A candidate correlated with default (but not pd_system) instead lets
# us test whether a genuine, non-mechanical risk signal survives.
# Both are useful; we report the full picture and choose in NB04b.

# %%
low_pd_candidates = corr_with_pd.head(15).index
default_corr = (
    graded[low_pd_candidates]
    .apply(lambda x: x.corr(graded[DEFAULT_COL], method="spearman"))
    .abs()
    .sort_values()
)

print("=== |corr| with default for the low-pd_system candidates ===")
print(default_corr.round(4).to_string())

# %% [markdown]
# ## 4. Joint summary table for candidate selection

# %%
summary = pd.DataFrame({
    "abs_corr_pd_system": corr_with_pd.head(15),
    "abs_corr_default"  : default_corr.reindex(low_pd_candidates),
}).sort_values("abs_corr_pd_system")

# Missing-value share matters too: a driver with heavy NaNs is unreliable
summary["missing_share"] = (
    graded[low_pd_candidates].isna().mean().reindex(summary.index).round(4)
)

print("=== Candidate selection table (sorted by corr with pd_system) ===")
print(summary.round(4).to_string())

print("\nGuidance:")
print("  * Primary driver  : low abs_corr_pd_system AND low missing_share")
print("  * For NB04b we will test two variants:")
print("      (b1) risk-neutral upgrades  -> also low abs_corr_default")
print("      (b2) risk-informed upgrades -> moderate abs_corr_default,")
print("           but still low abs_corr_pd_system")

Loaded: (43405, 70)
Columns: ['net_profit_to_assets', 'total_liabilities_to_assets', 'working_capital_to_assets', 'current_assets_to_short_liabilities', 'cash_to_current_liabilities', 'retained_earnings_to_assets', 'ebit_to_assets', 'book_value_equity_to_liabilities', 'sales_to_assets', 'equity_to_assets', 'Attr11', 'Attr12', 'Attr13', 'Attr14', 'Attr15', 'Attr16', 'Attr17', 'Attr18', 'Attr19', 'Attr20'] ...

Candidate financial variables: 58
=== LOWEST |corr| with pd_system (bottom 15) ===
-> best candidates for the independent DGM upgrade driver
Attr59    0.0011
Attr47    0.0165
Attr43    0.0267
Attr37    0.0618
Attr20    0.0799
Attr60    0.0833
Attr34    0.1216
Attr64    0.1235
Attr21    0.1416
Attr61    0.1458
Attr44    0.1517
Attr36    0.1628
Attr29    0.1895
Attr15    0.2043
Attr41    0.2253

=== HIGHEST |corr| with pd_system (top 10) ===
-> already embedded in pd_system; exclude as drivers
equity_to_assets    0.5924
Attr56              0.5973
Attr38              0.6187
Attr31   